In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

BASE = Path("../../..")
OUT_DIR = BASE / "data" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
rating_vars = {
    2010: [
        "contractid", "org_type", "contract_name", "org_marketing",
        "breastcancer_screen", "rectalcancer_screen", "cv_diab_cholscreen",
        "glaucoma_test", "monitoring", "flu_vaccine", "pn_vaccine",
        "physical_health", "mental_health", "osteo_test", "physical_monitor",
        "primaryaccess", "osteo_manage", "diab_healthy", "bloodpressure",
        "ra_manage", "copd_test", "bladder", "falling", "nodelays",
        "doctor_communicate", "carequickly", "customer_service",
        "overallrating_care", "overallrating_plan", "complaints_plan",
        "appeals_timely", "appeals_review", "leave_plan", "audit_problems",
        "hold_times", "info_accuracy", "ttyt_available",
    ],
    2011: [
        "contractid", "org_type", "contract_name", "org_marketing",
        "breastcancer_screen", "rectalcancer_screen", "cv_cholscreen",
        "diab_cholscreen", "glaucoma_test", "monitoring", "flu_vaccine",
        "pn_vaccine", "physical_health", "mental_health", "osteo_test",
        "physical_monitor", "primaryaccess", "osteo_manage", "diabetes_eye",
        "diabetes_kidney", "diabetes_bloodsugar", "diabetes_chol",
        "bloodpressure", "ra_manage", "copd_test", "bladder", "falling",
        "nodelays", "doctor_communicate", "carequickly", "customer_service",
        "overallrating_care", "overallrating_plan", "complaints_plan",
        "appeals_timely", "appeals_review", "corrective_action",
        "hold_times", "info_accuracy", "ttyt_available",
    ],
    2012: [
        "contractid", "org_type", "org_parent", "org_marketing",
        "breastcancer_screen", "rectalcancer_screen", "cv_cholscreen",
        "diab_cholscreen", "glaucoma_test", "flu_vaccine", "pn_vaccine",
        "physical_health", "mental_health", "physical_monitor", "primaryaccess",
        "bmi_assess", "older_medication", "older_function", "older_pain",
        "osteo_manage", "diabetes_eye", "diabetes_kidney", "diabetes_bloodsugar",
        "diabetes_chol", "bloodpressure", "ra_manage", "bladder", "falling",
        "readmissions", "nodelays", "carequickly", "customer_service",
        "overallrating_care", "overallrating_plan", "complaints_plan",
        "access_problems", "leave_plan", "appeals_timely", "appeals_review",
        "ttyt_available",
    ],
    2013: [
        "contractid", "org_type", "contract_name", "org_marketing", "org_parent",
        "breastcancer_screen", "rectalcancer_screen", "cv_cholscreen",
        "diab_cholscreen", "glaucoma_test", "flu_vaccine", "physical_health",
        "mental_health", "physical_monitor", "bmi_assess", "older_medication",
        "older_function", "older_pain", "osteo_manage", "diabetes_eye",
        "diabetes_kidney", "diabetes_bloodsugar", "diabetes_chol", "bloodpressure",
        "ra_manage", "bladder", "falling", "readmissions", "nodelays", "carequickly",
        "customer_service", "overallrating_care", "overallrating_plan",
        "coordination", "complaints_plan", "access_problems", "leave_plan",
        "improve", "appeals_timely", "appeals_review", "ttyt_available",
        "enroll_timely",
    ],
    2014: [
        "contractid", "org_type", "contract_name", "org_marketing", "org_parent",
        "breastcancer_screen", "rectalcancer_screen", "cv_cholscreen",
        "diab_cholscreen", "glaucoma_test", "flu_vaccine", "physical_health",
        "mental_health", "physical_monitor", "bmi_assess", "older_medication",
        "older_function", "older_pain", "osteo_manage", "diabetes_eye",
        "diabetes_kidney", "diabetes_bloodsugar", "diabetes_chol", "bloodpressure",
        "ra_manage", "bladder", "falling", "readmissions", "nodelays", "carequickly",
        "customer_service", "overallrating_care", "overallrating_plan",
        "coordination", "complaints_plan", "access_problems", "leave_plan",
        "improve", "appeals_timely", "appeals_review", "ttyt_available",
    ],
    2015: [
        "contractid", "org_type", "contract_name", "org_marketing", "org_parent",
        "rectalcancer_screen", "cv_cholscreen", "diab_cholscreen", "flu_vaccine",
        "physical_health", "mental_health", "physical_monitor", "bmi_assess",
        "specialneeds_manage", "older_medication", "older_function", "older_pain",
        "osteo_manage", "diabetes_eye", "diabetes_kidney", "diabetes_bloodsugar",
        "diabetes_chol", "bloodpressure", "ra_manage", "bladder", "falling",
        "readmissions", "nodelays", "carequickly", "customer_service",
        "overallrating_care", "overallrating_plan", "coordination",
        "complaints_plan", "leave_plan", "improve", "appeals_timely",
        "appeals_review",
    ],
}

In [3]:
star_file_config = {
    2010: dict(
        domain_file="2010_Part_C_Report_Card_Master_Table_2009_11_30_domain.csv",
        summary_file="2010_Part_C_Report_Card_Master_Table_2009_11_30_summary.csv",
        domain_skip=4,
        summary_skip=2,
        summary_cols=["contractid", "org_type", "contract_name",
                      "org_marketing", "partc_score"],
        domain_usecols=None,
        summary_usecols=None,
    ),
    2011: dict(
        domain_file="2011_Part_C_Report_Card_Master_Table_2011_04_20_star.csv",
        summary_file="2011_Part_C_Report_Card_Master_Table_2011_04_20_summary.csv",
        domain_skip=5,
        summary_skip=2,
        summary_cols=["contractid", "org_type", "contract_name", "org_marketing",
                      "partc_lowstar", "partc_score", "partcd_score"],
        domain_usecols=None,
        summary_usecols=None,
    ),
    2012: dict(
        domain_file="2012_Part_C_Report_Card_Master_Table_2011_11_01_Star.csv",
        summary_file="2012_Part_C_Report_Card_Master_Table_2011_11_01_Summary.csv",
        domain_skip=5,
        summary_skip=2,
        summary_cols=["contractid", "org_type", "org_parent", "org_marketing",
                      "partc_score", "partc_lowscore", "partc_highscore",
                      "partcd_score", "partcd_lowscore", "partcd_highscore"],
        domain_usecols=None,
        summary_usecols=None,
    ),
    2013: dict(
        domain_file="2013_Part_C_Report_Card_Master_Table_2012_10_17_Star.csv",
        summary_file="2013_Part_C_Report_Card_Master_Table_2012_10_17_Summary.csv",
        domain_skip=4,
        summary_skip=2,
        summary_cols=["contractid", "org_type", "org_marketing", "contract_name",
                      "org_parent", "partc_score", "partc_lowscore",
                      "partc_highscore", "partcd_score", "partcd_lowscore",
                      "partcd_highscore"],
        domain_usecols=None,
        summary_usecols=None,
    ),
    2014: dict(
        domain_file="2014_Part_C_Report_Card_Master_Table_2013_10_17_stars.csv",
        summary_file="2014_Part_C_Report_Card_Master_Table_2013_10_17_summary.csv",
        domain_skip=3,
        summary_skip=2,
        summary_cols=["contractid", "org_type", "org_marketing", "contract_name",
                      "org_parent", "snp", "sanction", "partc_score", "partcd_score"],
        domain_usecols=None,
        summary_usecols=list(range(9)),
    ),
    2015: dict(
        domain_file="2015_Report_Card_Master_Table_2014_10_03_stars.csv",
        summary_file="2015_Report_Card_Master_Table_2014_10_03_summary.csv",
        domain_skip=4,
        summary_skip=2,
        summary_cols=["contractid", "org_type", "org_marketing", "contract_name",
                      "org_parent", "snp", "sanction", "partc_score",
                      "partd_score", "partcd_score"],
        domain_usecols=list(range(38)),
        summary_usecols=list(range(10)),
    ),
}

In [7]:
# Read and build star ratings
def parse_number_like_readr(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return float("nan")
    s = str(x).strip()
    if s == "":
        return float("nan")
    s = s.replace(",", "")
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    return float(m.group(0)) if m else float("nan")


def build_year_star_ratings(year: int) -> pd.DataFrame:
    print(f"  Reading star ratings for {year}...")

    cfg = star_file_config[year]
    star_dir = BASE / "data" / "input" / f"star_ratings_{year}"

    domain_path  = star_dir / cfg["domain_file"]
    summary_path = star_dir / cfg["summary_file"]

    if not domain_path.exists():
        print(f"  ERROR: Domain file not found: {domain_path}")
        return pd.DataFrame()
    if not summary_path.exists():
        print(f"  ERROR: Summary file not found: {summary_path}")
        return pd.DataFrame()

    # --- Read domain file ---
    read_kwargs = dict(
        skiprows=cfg["domain_skip"],
        names=rating_vars[year],
        header=None,
        na_values=["", "NA", "*"],
        keep_default_na=True,
        encoding="latin1",  # <-- added
    )
    if cfg["domain_usecols"] is not None:
        read_kwargs["usecols"] = cfg["domain_usecols"]

    star_data_a = pd.read_csv(domain_path, **read_kwargs)

    exclude_cols = {"contractid", "org_type", "contract_name",
                    "org_marketing", "org_parent"}
    for c in [col for col in star_data_a.columns if col not in exclude_cols]:
        star_data_a[c] = star_data_a[c].map(
            parse_number_like_readr).astype("float64")

    # --- Read summary file ---
    read_kwargs_b = dict(
        skiprows=cfg["summary_skip"],
        names=cfg["summary_cols"],
        header=None,
        na_values=["", "NA", "*"],
        keep_default_na=True,
        encoding="latin1",  # <-- added
    )
    if cfg["summary_usecols"] is not None:
        read_kwargs_b["usecols"] = cfg["summary_usecols"]

    star_data_b = pd.read_csv(summary_path, **read_kwargs_b)

    # --- Process summary (logic differs by year) ---
    if year == 2010:
        star_data_b = star_data_b.assign(
            new_contract=lambda d: (
                d["partc_score"] == "Plan too new to be measured"
            ).astype("int64")
        )
        star_data_b["partc_score"] = star_data_b.apply(
            lambda r: float("nan") if r["new_contract"] == 1
            else parse_number_like_readr(r["partc_score"]), axis=1
        ).astype("float64")
        star_data_b = (
            star_data_b
            .loc[:, ["contractid", "new_contract", "partc_score"]]
            .assign(partcd_score=pd.NA)
        )

    elif year == 2011:
        star_data_b = star_data_b.assign(
            new_contract=lambda d: (
                (d["partc_score"] == "Plan too new to be measured") |
                (d["partcd_score"] == "Plan too new to be measured")
            ).astype("int64")
        )
        star_data_b["partc_score"] = star_data_b.apply(
            lambda r: float("nan") if r["new_contract"] == 1
            else parse_number_like_readr(r["partc_score"]), axis=1
        ).astype("float64")
        star_data_b["partcd_score"] = star_data_b.apply(
            lambda r: float("nan") if r["new_contract"] == 1
            else parse_number_like_readr(r["partcd_score"]), axis=1
        ).astype("float64")
        star_data_b["low_score"] = (
            star_data_b["partc_lowstar"] == "Yes"
        ).astype("float64")
        star_data_b = star_data_b.loc[
            :, ["contractid", "new_contract", "low_score",
                "partc_score", "partcd_score"]
        ]

    elif year in (2012, 2013):
        star_data_b = star_data_b.assign(
            new_contract=lambda d: (
                (d["partc_score"] == "Plan too new to be measured") |
                (d["partcd_score"] == "Plan too new to be measured")
            ).astype("int64")
        )
        star_data_b["partc_score"] = star_data_b.apply(
            lambda r: float("nan") if r["new_contract"] == 1
            else parse_number_like_readr(r["partc_score"]), axis=1
        ).astype("float64")
        star_data_b["partcd_score"] = star_data_b.apply(
            lambda r: float("nan") if r["new_contract"] == 1
            else parse_number_like_readr(r["partcd_score"]), axis=1
        ).astype("float64")
        star_data_b["low_score"] = (
            star_data_b["partc_lowscore"] == "Yes"
        ).astype("float64")
        star_data_b = star_data_b.loc[
            :, ["contractid", "new_contract", "low_score",
                "partc_score", "partcd_score"]
        ]

    elif year in (2014, 2015):
        star_data_b = star_data_b.assign(
            new_contract=lambda d: (
                (d["partc_score"] == "Plan too new to be measured") |
                (d["partcd_score"] == "Plan too new to be measured")
            ).astype("int64")
        )
        star_data_b["partc_score"] = star_data_b.apply(
            lambda r: float("nan") if r["new_contract"] == 1
            else parse_number_like_readr(r["partc_score"]), axis=1
        ).astype("float64")
        star_data_b["partcd_score"] = star_data_b.apply(
            lambda r: float("nan") if r["new_contract"] == 1
            else parse_number_like_readr(r["partcd_score"]), axis=1
        ).astype("float64")
        star_data_b = star_data_b.loc[
            :, ["contractid", "new_contract", "partc_score", "partcd_score"]
        ]

    # --- Merge domain + summary ---
    drop_cols = ["contract_name", "org_type", "org_marketing", "org_parent"]
    final_star = (
        star_data_a
        .drop(columns=drop_cols, errors="ignore")
        .merge(star_data_b, on="contractid", how="left")
        .assign(year=year)
    )

    print(f"  Star ratings: {len(final_star):,} contracts")
    return final_star

In [8]:
# Merge star ratings into penetration data
def build_year_final(year: int) -> pd.DataFrame:
    pen_path = OUT_DIR / f"ma_data_{year}_with_penetration.csv"
    if not pen_path.exists():
        print(f"ERROR: Penetration file not found for {year}.")
        return pd.DataFrame()

    df = pd.read_csv(pen_path, low_memory=False)
    print(f"\n{'='*60}")
    print(f"Adding star ratings for {year} ({len(df):,} rows)")
    print(f"{'='*60}")

    star_df = build_year_star_ratings(year)
    if star_df.empty:
        return df

    # Star ratings are contract-level, merge on contractid only
    merged = df.merge(
        star_df.drop(columns=["year"], errors="ignore"),
        on="contractid",
        how="left"
    )

    # Compute Star_Rating exactly as professor's code
    partd  = merged.get("partd",        pd.Series(dtype=str))
    partc  = merged.get("partc_score",  pd.Series(dtype=float))
    partcd = merged.get("partcd_score", pd.Series(dtype=float))

    merged["Star_Rating"] = np.where(
        partd == "No", partc,
        np.where(
            (partd == "Yes") & partcd.isna(), partc,
            np.where(
                (partd == "Yes") & partcd.notna(), partcd,
                np.nan
            )
        )
    )

    print(f"  After star ratings merge: {len(merged):,} rows")
    print(f"  Plans with Star_Rating:    {merged['Star_Rating'].notna().sum():,}")
    print(f"  Plans without Star_Rating: {merged['Star_Rating'].isna().sum():,}")

    out_path = OUT_DIR / f"ma_data_{year}_final.csv"
    merged.to_csv(out_path, index=False)
    print(f"  Saved to: {out_path}")

    return merged

In [9]:
# Run for all years
def build_all_years_final(years: list = [2010, 2011, 2012, 2013, 2014, 2015]):
    results = {}

    for year in years:
        try:
            df = build_year_final(year)
            results[year] = df
        except Exception as e:
            print(f"\n✗ ERROR processing {year}: {e}")
            results[year] = pd.DataFrame()

    print(f"\n{'='*60}")
    print("Summary:")
    print(f"{'='*60}")
    for year, df in results.items():
        status = "✓" if not df.empty else "✗"
        n_rated = df["Star_Rating"].notna().sum() if not df.empty else 0
        print(f"{status} {year}: {len(df):,} rows | {n_rated:,} with Star_Rating")

    return results

results_final = build_all_years_final()


Adding star ratings for 2010 (112,856 rows)
  Reading star ratings for 2010...
  Star ratings: 709 contracts
  After star ratings merge: 112,856 rows
  Plans with Star_Rating:    63,320
  Plans without Star_Rating: 49,536
  Saved to: ../../../data/output/ma_data_2010_final.csv

Adding star ratings for 2011 (70,184 rows)
  Reading star ratings for 2011...
  Star ratings: 575 contracts
  After star ratings merge: 70,184 rows
  Plans with Star_Rating:    57,421
  Plans without Star_Rating: 12,763
  Saved to: ../../../data/output/ma_data_2011_final.csv

Adding star ratings for 2012 (69,770 rows)
  Reading star ratings for 2012...
  Star ratings: 569 contracts
  After star ratings merge: 69,770 rows
  Plans with Star_Rating:    61,027
  Plans without Star_Rating: 8,743
  Saved to: ../../../data/output/ma_data_2012_final.csv

Adding star ratings for 2013 (70,930 rows)
  Reading star ratings for 2013...
  Star ratings: 579 contracts
  After star ratings merge: 70,930 rows
  Plans with Star_R

In [10]:
df = pd.read_csv("../../../data/output/ma_data_2010_final.csv", low_memory=False)
measure_cols = [c for c in rating_vars[2010] 
                if c not in {"contractid","org_type","contract_name",
                             "org_marketing","org_parent"}]
print("Measure cols present:", [c for c in measure_cols if c in df.columns])
print("Measure cols missing:", [c for c in measure_cols if c not in df.columns])

Measure cols present: ['breastcancer_screen', 'rectalcancer_screen', 'cv_diab_cholscreen', 'glaucoma_test', 'monitoring', 'flu_vaccine', 'pn_vaccine', 'physical_health', 'mental_health', 'osteo_test', 'physical_monitor', 'primaryaccess', 'osteo_manage', 'diab_healthy', 'bloodpressure', 'ra_manage', 'copd_test', 'bladder', 'falling', 'nodelays', 'doctor_communicate', 'carequickly', 'customer_service', 'overallrating_care', 'overallrating_plan', 'complaints_plan', 'appeals_timely', 'appeals_review', 'leave_plan', 'audit_problems', 'hold_times', 'info_accuracy', 'ttyt_available']
Measure cols missing: []
